# The Weekend Holds Along the Coast

Monterey Bay has one busy whale-watching fleet. Does the weekend pattern show up along the rest of the
West Coast too? `dim_location.area` splits the box at Point Conception (34.45 N), the Oregon border
(42 N) and the Columbia River (46.25 N), with Monterey Bay taken out of northern California. The
weekend ratio is the same as in notebook 01.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## Weekend Ratio for Each Area and Kind of Source
Areas north to south; kinds with at least 200 records.

In [ ]:
%%sql
SELECT
    dim_location.area,
    dim_dataset.kind,
    COUNT(*) AS records,
    ROUND(
        (COUNT_IF(dim_date.is_weekend) / 2) / NULLIF(COUNT_IF(dim_date.is_midweek) / 3, 0),
        2
    ) AS weekend_ratio
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
GROUP BY dim_location.area, dim_location.area_order, dim_dataset.kind
HAVING COUNT(*) >= 200
ORDER BY dim_location.area_order, dim_dataset.kind

## How Sure? Resample Weeks in Each Area
The public ratio in each area, with a 95% range as in notebook 01. Each area draws its own weeks, so the Monterey Bay range can differ by a hundredth from the one in notebook 01.

In [ ]:
%%sql
WITH weekly_counts AS (
    SELECT
        dim_location.area,
        dim_location.area_order,
        DATEADD(DAY, 1 - dim_date.weekday_number, dim_date.date) AS week_start,
        COUNT_IF(dim_date.is_weekend) AS weekend_records,
        COUNT_IF(dim_date.is_midweek) AS midweek_records
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_dataset.kind = 'public'
    GROUP BY dim_location.area, dim_location.area_order, week_start
),

numbered_weeks AS (
    SELECT
        area,
        area_order,
        ROW_NUMBER() OVER (PARTITION BY area ORDER BY week_start) AS week_number,
        COUNT(*) OVER (PARTITION BY area) AS total_weeks,
        weekend_records,
        midweek_records
    FROM weekly_counts
),

replicates AS (
    SELECT SEQ4() AS replicate
    FROM TABLE(GENERATOR(ROWCOUNT => 2000))
),

-- In each area, each replicate draws as many weeks as the area has,
-- at random and with replacement.
resample_draws AS (
    SELECT
        replicates.replicate,
        numbered_weeks.area,
        1 + MOD(
            ABS(HASH(replicates.replicate, numbered_weeks.area, numbered_weeks.week_number, 20261016)),
            numbered_weeks.total_weeks
        ) AS week_number
    FROM replicates
    CROSS JOIN numbered_weeks
),

resampled_ratios AS (
    SELECT
        resample_draws.area,
        resample_draws.replicate,
        (SUM(numbered_weeks.weekend_records) / 2)
        / NULLIF(SUM(numbered_weeks.midweek_records) / 3, 0) AS weekend_ratio
    FROM resample_draws
    JOIN numbered_weeks
      ON numbered_weeks.area = resample_draws.area
     AND numbered_weeks.week_number = resample_draws.week_number
    GROUP BY resample_draws.area, resample_draws.replicate
),

observed_ratios AS (
    SELECT
        area,
        ANY_VALUE(area_order) AS area_order,
        (SUM(weekend_records) / 2) / NULLIF(SUM(midweek_records) / 3, 0) AS weekend_ratio
    FROM numbered_weeks
    GROUP BY area
)

SELECT
    observed_ratios.area,
    ROUND(observed_ratios.weekend_ratio, 2) AS weekend_ratio,
    ROUND(PERCENTILE_CONT(0.025) WITHIN GROUP (ORDER BY resampled_ratios.weekend_ratio), 2)
        AS range_low,
    ROUND(PERCENTILE_CONT(0.975) WITHIN GROUP (ORDER BY resampled_ratios.weekend_ratio), 2)
        AS range_high
FROM observed_ratios
JOIN resampled_ratios ON resampled_ratios.area = observed_ratios.area
GROUP BY observed_ratios.area, observed_ratios.area_order, observed_ratios.weekend_ratio
ORDER BY observed_ratios.area_order

## Result
Public sightings rise at weekends in **4 of 5 areas**: a weekend day carries 1.44× the public sightings of a
midweek day in Washington and British Columbia, 1.85× in northern California outside the bay, 1.52× in
Monterey Bay and 1.83× in southern California. Oregon, with the fewest public records, shows none: a weekend
day carries 0.87× the public sightings of a midweek day, and the 95% range (0.67 to 1.16) includes 1, no
difference at all.

## Limits
* Monterey has an unusually large whale-watching fleet; this comparison only sketches the effect elsewhere.
* Puget Sound's main sighting network is not in the OBIS archive, so Washington's public records are
  undercounted.